# **Fase 3a · Transcrição automática**
### *Executa o Whisper large-v3 nas cinco faixas, nas quatro condições definidas em `docs/fase3/01_desenho_experimental.md`, e salva cada execução em JSON. Requer acesso aos áudios e GPU.*

(Nenhuma medida é calculada neste caderno. A taxa de erro de palavra e a classificação dos termos críticos estão no caderno 3b, que lê apenas os JSON produzidos aqui e as transcrições de referência da fase 1)

## **1. Preparação do ambiente**

*Instala as duas bibliotecas que o Colab não traz, o Whisper e o filtro de ruído, com as versões fixadas. O caderno exige GPU, porque o Whisper large-v3 tem cerca de 1,5 bilhão de parâmetros, e o volume de cálculo tornaria a execução em processador comum lenta demais.*

In [ ]:
!pip install -q openai-whisper==20250625 noisereduce==3.0.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 21.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


### 1.1 Drive e pastas

Conecta o Google Drive, define a pasta dos áudios e cria a pasta onde os resultados são gravados (`resultados/fase3`). Os JSON são salvos primeiro no Drive, que não se apaga quando a sessão do Colab cai, e só ao fim da fase são levados ao repositório. A lista impressa confere que a numeração das faixas é a mesma da fase 2.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PASTA_PROJETO = '/content/drive/MyDrive/escuta-cerrada-poc'
PASTA_AUDIOS = os.path.join(PASTA_PROJETO, 'audios')
PASTA_RESULTADOS = os.path.join(PASTA_PROJETO, 'resultados', 'fase3')
os.makedirs(PASTA_RESULTADOS, exist_ok=True)

ARQUIVOS = sorted(os.listdir(PASTA_AUDIOS))
for i, nome in enumerate(ARQUIVOS, start=1):
    print(i, nome)

Mounted at /content/drive
1 1. 27230 - Concentração trabalhista no estádio do Pacaembú - 01-05-1944.mp3
2 2. Dia de Verão Gaúcho.wav
3 3. 27240 - Doutrina policial - é a prostituição um mal necessário - 1944 - Experiência de gravação (B).mp3
4 4. 27698 - Ritmos de samba - Almirante.wav
5 5. Descobrimento do Brasil.mp4


### 1.2 Versões e GPU

Registra as versões do Python e das bibliotecas e confere se a GPU está ativa. O PyTorch (`torch`) é a biblioteca sobre a qual o Whisper funciona e que se comunica com a GPU.

In [ ]:
import sys, importlib.metadata as md, torch

print('Python', sys.version.split()[0])
for pacote in ['openai-whisper', 'noisereduce', 'librosa', 'numpy', 'torch']:
    print(pacote, md.version(pacote))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nenhuma')

Python 3.13.15
openai-whisper 20250625
noisereduce 3.0.3
librosa 0.11.0
numpy 2.1.3
torch 2.11.0+cu130
GPU: Tesla T4


**Resultado.** GPU Tesla T4 ativa. O Python está na versão 3.13.15, enquanto na fase 2 estava na 3.13.16, porque o Colab atualiza o ambiente sem aviso. Por esse motivo, cada JSON registra as versões em uso no momento da execução.

### 1.3 Avisos silenciados

Silencia dois avisos do librosa que aparecem ao ler o MP4 da faixa 05, cuja leitura recorre a uma biblioteca alternativa. Os avisos não indicam erro. Na execução original, esta célula foi rodada depois da seção 2, de modo que os avisos aparecem na saída da seção 2.2.

In [2]:
import warnings
warnings.filterwarnings('ignore', message='PySoundFile failed')
warnings.filterwarnings('ignore', category=FutureWarning)

## **2. O áudio na forma em que o modelo o recebe**

*Carrega cada faixa como soma dos canais dividida por dois, decisão registrada no diário da fase 2, e a reamostra para 16 kHz, taxa em que o Whisper foi treinado. Uma taxa de amostragem só representa frequências até a metade do seu valor, de modo que a reamostragem descarta tudo acima de 8 kHz. Nessas gravações, isso atinge apenas o ruído da digitalização, já que a voz não ultrapassa cerca de 4 kHz, mas constitui um segundo filtro implícito na entrada do modelo, ao lado da soma dos canais.*

In [3]:
import librosa
import numpy as np

TAXA_WHISPER = 16000

def carregar_16k(n):
    caminho = os.path.join(PASTA_AUDIOS, ARQUIVOS[n - 1])
    sinal, _ = librosa.load(caminho, sr=TAXA_WHISPER, mono=False)
    if sinal.ndim == 2:
        mono = (sinal[0] + sinal[1]) / 2
    else:
        mono = sinal
    return mono.astype(np.float32)

### 2.1 Conferência das durações

In [ ]:
for n in range(1, 6):
    audio = carregar_16k(n)
    print(n, f'{len(audio) / TAXA_WHISPER:.1f} s')

1 335.5 s
2 201.3 s
3 362.1 s
4 64.9 s
5 441.3 s


**Resultado.** As durações coincidem com as medidas na fase 2 (335,5, 201,3, 362,1, 64,9 e 441,3 segundos), o que confirma que a reamostragem não cortou nem alterou a extensão de nenhuma faixa.

## **3. Filtro de ruído**

*Define o filtro usado nas condições C e D, a função `reduce_noise` da biblioteca noisereduce com os parâmetros padrão. A técnica (spectral gating, ou "portão espectral") divide o som em faixas de frequência, estima o nível de ruído em cada uma e atenua o que fica abaixo desse nível. Na configuração padrão, não estacionária, o nível de ruído é reestimado continuamente ao longo da gravação. O filtro não distingue som com sentido de ruído, apenas o que é fraco e constante do que se destaca.*

In [ ]:
import noisereduce as nr

def filtrar(audio):
    return nr.reduce_noise(y=audio, sr=TAXA_WHISPER).astype(np.float32)

### 3.1 Escuta do filtro · faixa 03, de 4:13 a 4:19 (virada de página)

Toca o trecho como o modelo o recebe na condição sem filtro e, em seguida, com o filtro. Aplicado a seis segundos, o filtro estima o ruído apenas a partir desse trecho, e o efeito sobre a faixa inteira pode ser um pouco diferente. As saídas com os tocadores foram removidas antes da publicação.

In [ ]:
from IPython.display import Audio, display

trecho = carregar_16k(3)[253 * TAXA_WHISPER : 259 * TAXA_WHISPER]
print('Original (soma, 16 kHz)')
display(Audio(trecho, rate=TAXA_WHISPER))
print('Com filtro')
display(Audio(filtrar(trecho), rate=TAXA_WHISPER))

**Resultado.** Com o filtro, o chiado desaparece e o trecho soa "limpo". Os ruídos do papel continuam presentes, mas muito atenuados e com timbre estranho, a ponto de não serem mais reconhecíveis como virada de página. O som de papel é de banda larga e sem altura definida, semelhante ao próprio ruído de superfície, e o portão espectral corta as partes de seu espectro que se confundem com o ruído, deixando passar fragmentos que soam metálicos, artefato conhecido como ruído musical (musical noise). O evento permanece como energia, mas perde a qualidade pela qual a escuta o reconhece.

## **4. Modelo, glossário e condições**

*Carrega o Whisper large-v3 na GPU. O download, de cerca de 3 GB, corresponde aos pesos do modelo, os parâmetros aprendidos no treinamento, e ocorre uma vez por sessão.*

In [ ]:
import whisper
modelo = whisper.load_model('large-v3', device='cuda')

100%|██████████████████████████████████████| 2.88G/2.88G [00:29<00:00, 106MiB/s]


### 4.1 Glossário e condições

Transcreve para o código as seções 2 e 4 do desenho experimental. As duas variáveis atuam em pontos diferentes. O filtro altera o próprio áudio antes de o modelo recebê-lo, enquanto o glossário deixa o áudio intacto e altera a expectativa linguística do modelo, por meio de um texto (prompt inicial) que ele recebe antes de transcrever e que inclina suas escolhas na direção daquelas palavras. O glossário não retreina o modelo, cujos pesos permanecem os mesmos.

In [ ]:
GLOSSARIO = {
    1: 'Pacaembu, Dia do Trabalho, sindicalização.',
    2: 'Madrugada, Meio-dia, Tardezinha.',
    3: 'prostituição, mal necessário.',
    4: 'batucada, surdo, tamborim, primeiro tamborim, segundo tamborim.',
    5: 'Dom Aires Correia, batel, Portugal, Cabral, Vera Cruz.',
}

CONDICOES = {
    'A': {'glossario': False, 'filtro': False},
    'B': {'glossario': True,  'filtro': False},
    'C': {'glossario': False, 'filtro': True},
    'D': {'glossario': True,  'filtro': True},
}

SEMENTE = 0

## **5. Função de execução e registro**

*Define a função que executa uma condição sobre uma faixa e salva o resultado. Ela prepara a entrada (carrega a faixa, aplica o filtro se a condição o prevê e escolhe o prompt), fixa a semente aleatória, transcreve com os parâmetros do desenho experimental e grava um JSON com o texto, os segmentos, as palavras com marcação de tempo e probabilidade, os parâmetros, as versões, a GPU, a data e a duração do processamento.*

*A semente importa por causa do mecanismo de recurso do Whisper. Cada trecho é transcrito primeiro na forma mais provável, sem sorteio (temperatura 0). Se o resultado parecer defeituoso, por exemplo com repetição excessiva ou confiança muito baixa, o modelo tenta de novo com temperaturas mais altas, em que passa a sortear entre palavras possíveis. A semente fixa esse sorteio, para que se repita a cada execução.*

In [ ]:
import json, time, random
from datetime import datetime, timezone

def executar(n, condicao, sufixo=''):
    cfg = CONDICOES[condicao]
    audio = carregar_16k(n)
    if cfg['filtro']:
        audio = filtrar(audio)
    prompt = GLOSSARIO[n] if cfg['glossario'] else None

    random.seed(SEMENTE)
    np.random.seed(SEMENTE)
    torch.manual_seed(SEMENTE)

    inicio = time.time()
    saida = modelo.transcribe(
        audio,
        language='pt',
        word_timestamps=True,
        initial_prompt=prompt,
        carry_initial_prompt=prompt is not None,
    )
    duracao = time.time() - inicio

    registro = {
        'faixa': n,
        'arquivo': ARQUIVOS[n - 1],
        'condicao': condicao,
        'glossario': cfg['glossario'],
        'filtro': cfg['filtro'],
        'prompt_inicial': prompt,
        'modelo': 'large-v3',
        'parametros': {
            'language': 'pt',
            'word_timestamps': True,
            'carry_initial_prompt': prompt is not None,
            'demais': 'valores padrão da biblioteca',
        },
        'semente': SEMENTE,
        'versoes': {p: md.version(p) for p in
                    ['openai-whisper', 'noisereduce', 'librosa', 'numpy', 'torch']},
        'python': sys.version.split()[0],
        'gpu': torch.cuda.get_device_name(0),
        'data_execucao': datetime.now(timezone.utc).isoformat(),
        'duracao_audio_s': round(len(audio) / TAXA_WHISPER, 2),
        'duracao_processamento_s': round(duracao, 2),
        'texto': saida['text'],
        'segmentos': saida['segments'],
    }

    nome = f'faixa{n:02d}_{condicao}{sufixo}.json'
    with open(os.path.join(PASTA_RESULTADOS, nome), 'w', encoding='utf-8') as f:
        json.dump(registro, f, ensure_ascii=False, indent=2, default=float)
    print(f'{nome} salvo · {duracao:.0f} s de processamento')
    return registro

## **6. Primeira execução e teste de determinismo**

*Executa a condição A sobre a faixa 04, a mais curta. É ao mesmo tempo a primeira das vinte execuções e a primeira metade do teste de determinismo. Ao ser reexecutada, a célula sobrescreve `faixa04_A.json` com resultado idêntico (seção 6.2).*

In [ ]:
r = executar(4, 'A')
print(r['texto'])

faixa04_A.json salvo · 16 s de processamento
 Aqui está uma demonstração de ritmo de samba do Brasil. Vamos mostrar aqui todos os instrumentos que entram no samba. Iniciando pela batucada. Este é o surdo. Enquanto o surdo faz isso, o primeiro tamborim faz isso, vocês vão ouvir. Esse tamborim vem um segundo tamborim fazendo um ritmo diferente assim. Assim o tamborim, já depois o pano de som. E assim o tamborim.


**Resultado.** Três pontos ficam registrados como observações preliminares, a conferir na medição e na escuta dirigida. A palavra "principiando" foi transcrita como "iniciando", troca por um sinônimo mais frequente hoje. No lugar da palavra perdida marcada na referência ("depois do [?] [...] de som"), o modelo escreveu "o pano de som". E a frase final, "E assim o tamborim.", não tem correspondente na referência, podendo ser fala não transcrita ou invenção num trecho só de percussão.

### 6.1 Verificação da pasta de resultados

Lista o conteúdo da pasta de resultados diretamente pelo Colab. A interface web do Drive pode demorar a exibir arquivos gravados pelo Colab.

In [ ]:
print(PASTA_RESULTADOS)
for nome in sorted(os.listdir(PASTA_RESULTADOS)):
    print(nome)

/content/drive/MyDrive/escuta-cerrada-poc/resultados/fase3
faixa04_A.json


### 6.2 Teste de determinismo

Executa de novo a condição A sobre a faixa 04, salvando com outro nome, e compara as duas execuções em três níveis (texto; texto e tempos; tudo, inclusive probabilidades). Antes da execução ficou decidido que texto e tempos idênticos bastam para considerar a execução determinística, já que o WER e os termos críticos dependem apenas do texto. Diferenças apenas nas casas decimais das probabilidades seriam registradas, sem acionar as três repetições previstas no desenho experimental.

In [ ]:
r2 = executar(4, 'A', sufixo='_repeticao')

def sem_probabilidades(segmentos):
    return [
        {'inicio': s['start'], 'fim': s['end'], 'texto': s['text'],
         'palavras': [(p['word'], p['start'], p['end']) for p in s['words']]}
        for s in segmentos
    ]

print('Texto idêntico:', r['texto'] == r2['texto'])
print('Texto e tempos idênticos:',
      sem_probabilidades(r['segmentos']) == sem_probabilidades(r2['segmentos']))
print('Tudo idêntico, inclusive probabilidades:',
      json.dumps(r['segmentos'], default=float) == json.dumps(r2['segmentos'], default=float))

faixa04_A_repeticao.json salvo · 9 s de processamento
Texto idêntico: True
Texto e tempos idênticos: True
Tudo idêntico, inclusive probabilidades: True


**Resultado.** As duas execuções são idênticas nos três níveis, até a última casa decimal das probabilidades. Pela regra do desenho experimental, cada condição roda uma vez. O teste cobriu apenas a condição A na faixa 04, e a repetição nos trechos em que o modelo recorre ao sorteio depende da semente, sem ter sido testada diretamente.

A primeira execução levou 16 segundos, e a repetição, 9. A diferença se deve ao aquecimento da GPU, que na primeira chamada da sessão aloca memória e prepara as operações, e deve ser considerada em qualquer leitura dos tempos de processamento.

## **7. As vinte execuções**

*Percorre as quatro condições e as cinco faixas. Antes de cada execução, verifica se o arquivo correspondente já existe e, nesse caso, pula a combinação, o que preserva a execução já feita da faixa 04 e permite retomar o processo de onde parou se a sessão cair.*

In [ ]:
for condicao in ['A', 'B', 'C', 'D']:
    for n in range(1, 6):
        nome = f'faixa{n:02d}_{condicao}.json'
        if os.path.exists(os.path.join(PASTA_RESULTADOS, nome)):
            print(f'{nome} já existe, pulando')
            continue
        executar(n, condicao)

faixa01_A.json salvo · 151 s de processamento
faixa02_A.json salvo · 95 s de processamento
faixa03_A.json salvo · 97 s de processamento
faixa04_A.json já existe, pulando
faixa05_A.json salvo · 31 s de processamento
faixa01_B.json salvo · 80 s de processamento
faixa02_B.json salvo · 36 s de processamento
faixa03_B.json salvo · 144 s de processamento
faixa04_B.json salvo · 9 s de processamento
faixa05_B.json salvo · 20 s de processamento
faixa01_C.json salvo · 67 s de processamento
faixa02_C.json salvo · 46 s de processamento
faixa03_C.json salvo · 69 s de processamento
faixa04_C.json salvo · 8 s de processamento
faixa05_C.json salvo · 123 s de processamento
faixa01_D.json salvo · 99 s de processamento
faixa02_D.json salvo · 36 s de processamento
faixa03_D.json salvo · 111 s de processamento
faixa04_D.json salvo · 6 s de processamento
faixa05_D.json salvo · 19 s de processamento


**Resultado.** As vinte execuções foram concluídas em cerca de meia hora. Os tempos de processamento variam muito mais do que as durações dos áudios explicariam (a faixa 05 levou 31 segundos na condição A e 123 na condição C), porque dependem sobretudo da quantidade de texto produzida e do número de novas tentativas com sorteio.

## **8. Visão geral das execuções**

*Reúne numa tabela, para cada execução, o número de palavras escritas, o número de segmentos, quantos segmentos recorreram ao sorteio (temperatura acima de 0), quantos têm texto apesar de o modelo estimar mais de 60% de probabilidade de não haver fala (`no_speech_prob` acima de 0,6) e o tempo de processamento. Para comparação, as transcrições de referência têm cerca de 468, 331, 663, 61 e 509 palavras nas faixas 01 a 05, contagem aproximada que inclui as marcações.*

In [ ]:
import pandas as pd

linhas = []
for nome in sorted(os.listdir(PASTA_RESULTADOS)):
    if not nome.endswith('.json') or 'repeticao' in nome:
        continue
    with open(os.path.join(PASTA_RESULTADOS, nome), encoding='utf-8') as f:
        r = json.load(f)
    segs = r['segmentos']
    linhas.append({
        'faixa': r['faixa'],
        'cond.': r['condicao'],
        'palavras': len(r['texto'].split()),
        'segmentos': len(segs),
        'com sorteio': sum(s['temperature'] > 0 for s in segs),
        'sem fala?': sum(s['no_speech_prob'] > 0.6 for s in segs),
        'proc. (s)': r['duracao_processamento_s'],
    })

tabela = pd.DataFrame(linhas)
print(tabela.to_string(index=False))

 faixa cond.  palavras  segmentos  com sorteio  sem fala?  proc. (s)
     1     A       448         63           40          0     151.11
     1     B       192         25            7          0      79.76
     1     C       435         54            6          0      67.11
     1     D       424         50           15          0      99.15
     2     A       393         40           21          6      94.79
     2     B       344         35            0          0      35.58
     2     C       359         49           12          0      46.49
     2     D       344         35            0          0      35.99
     3     A       678         62           10          0      97.33
     3     B       731         74           31          0     143.82
     3     C       668         56            0          0      69.25
     3     D       690         59            7          0     110.84
     4     A        64          8            1          0      16.43
     4     B        62          8 

**Resultado.** A leitura dos segmentos de cada JSON, feita a partir desta tabela, mostrou os seguintes pontos, registrados como preliminares até a medição (caderno 3b) e a escuta dirigida.

Na faixa 05, nas condições A, B e D, o modelo escreveu "A CIDADE NO BRASIL" 15 ou 16 vezes seguidas, cobrindo quase seis minutos de diálogo. A frase surge aos 25 segundos, durante a música do início, em que não há fala. Só a condição C escapou, com "E aí" repetido na parte musical e início da transcrição aos 111 segundos, coincidindo com a entrada da voz registrada na escuta cega (1:52).

Na faixa 01, na condição B, o texto segue a fala até 1:29, passa por um trecho sem sentido que mistura português, inglês e caracteres árabes e, a partir daí, repete "Ação da Riqueza Nacional." oito vezes até o fim, frase derivada de "expansão da riqueza nacional", presente no discurso. Na faixa 02, condição C, há um loop menor, com "O fim do mar" repetido sete vezes.

Os casos correspondem ao fenômeno descrito como alucinação em loop. O Whisper transcreve em trechos de 30 segundos e, por padrão, cada trecho recebe como contexto o texto do anterior, de modo que uma invenção num trecho difícil tende a se propagar pelos seguintes. Nos segmentos "A CIDADE NO BRASIL", a probabilidade de não haver fala é baixa e a confiança média é alta, ou seja, os indicadores de confiança do próprio modelo não sinalizam a invenção.

Os efeitos das variáveis não são fixos. Na faixa 05, o filtro sozinho evita o loop, mas filtro e glossário juntos não evitam. Na faixa 01, o glossário sozinho provoca o colapso, mas glossário e filtro juntos não provocam. As execuções são determinísticas, mas não estáveis, já que uma pequena diferença no ponto de partida leva a trajetórias inteiramente distintas, e a comparação entre condições será lida com essa ressalva.

Outros pontos para a medição e a escuta dirigida: na faixa 01, condição A, o modelo escreveu "Aplausos." três vezes (por volta de 4:12, 4:55 e 5:27), convenção de legendagem aprendida nos dados de treino; numa busca no texto bruto, nenhuma execução escreveu "Pacaembu", "Dia do Trabalho", "sindicalização" ou "Vera Cruz"; na faixa 05, condição C, aparece "Vem doar-lhe as correias", provável deformação de "Dom Aires Correia".

## **9. Escuta dirigida**

*Toca cada trecho da lista gerada pelo caderno 3b (seção 7), a partir do arquivo original, com três segundos de folga antes e depois. Por padrão, o texto da máquina não é exibido, para que a primeira escuta não seja orientada por ele. As saídas com tocadores são removidas antes da publicação.*

In [4]:
import pandas as pd
from IPython.display import Audio, display

ESCUTA = pd.read_csv(os.path.join(PASTA_RESULTADOS, 'medidas', 'escuta_dirigida.csv'))

def ouvir_trecho(t, mostrar_texto=False, margem=3):
    g = ESCUTA[ESCUTA['trecho'] == t]
    n = int(g['faixa'].iloc[0])
    inicio = max(0, g['inicio'].min() - margem)
    fim = g['fim'].max() + margem
    caminho = os.path.join(PASTA_AUDIOS, ARQUIVOS[n - 1])
    sinal, taxa = librosa.load(caminho, sr=None, mono=False, offset=inicio, duration=fim - inicio)
    mono = (sinal[0] + sinal[1]) / 2 if sinal.ndim == 2 else sinal
    print(f'Trecho {t} · faixa {n:02d} · {int(inicio // 60)}:{inicio % 60:04.1f} a {int(fim // 60)}:{fim % 60:04.1f}')
    if mostrar_texto:
        for _, l in g.iterrows():
            print(f"   {l['cond.']} · {l['criterio']}: {l['texto']}")
    display(Audio(mono, rate=taxa))

Cada trecho é tocado trocando o número na chamada abaixo, de 1 a 31. A primeira escuta é feita sem o texto da máquina, e a segunda com `ouvir_trecho(n, mostrar_texto=True)` ou com o texto do caderno de preenchimento. O registro da escuta está em `docs/fase3/02_escuta_dirigida.md`.

In [ ]:
ouvir_trecho(1)

### 9.1 Escuta complementar · os segmentos "Aplausos." da faixa 01

Na condição A, o modelo escreveu "Aplausos." em três segmentos isolados da faixa 01. Por terem uma palavra só, esses segmentos não entraram na lista da escuta dirigida. A célula toca seis segundos em torno de cada um, três antes e três depois do tempo marcado pelo modelo.

In [ ]:
def ouvir(n, inicio, fim):
    caminho = os.path.join(PASTA_AUDIOS, ARQUIVOS[n - 1])
    sinal, taxa = librosa.load(caminho, sr=None, mono=False, offset=inicio, duration=fim - inicio)
    mono = (sinal[0] + sinal[1]) / 2 if sinal.ndim == 2 else sinal
    display(Audio(mono, rate=taxa))

for t in [252.2, 295.2, 327.3]:
    print(f'Faixa 01 · {int(t // 60)}:{t % 60:04.1f}')
    ouvir(1, t - 3, t + 3)

**Resultado.** Escuta feita em 13/10/2026. Nos três casos soam aplausos. Em 4:12,2 eles começam exatamente no tempo marcado pelo modelo, em 4:55,2 já estão em curso, e em 5:27,3 começam por volta de 5:27,5. A referência registra sete eventos de aplauso na faixa 01, e o modelo de fala anotou três deles, apenas na condição A, sem nenhum aplauso escrito onde não havia aplauso.